# Практична робота №7 — Цілочислове та булеве програмування: метод гілок і меж

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою задачу ЦЛП і дані рюкзака автоматично.

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Постановка задачі ЦЛП і неперервне ослаблення

Z = c1·x1 + c2·x2 → max за двома обмеженнями, x1,x2 — цілі невід'ємні. Спочатку розв'язуємо задачу БЕЗ вимоги цілочисловості (ослаблення) — це дає верхню межу для будь-якого цілого розв'язку.

In [ ]:
c1, c2, a1, b1, a2, b2, n_items, W = VARIANTS[VARIANT]
c = [c1, c2]
A = [list(a1), list(a2)]
b = [b1, b2]
c_min = [-c1, -c2]  # linprog мінімізує

res_lp = linprog(c_min, A_ub=A, b_ub=b, bounds=[(0, None), (0, None)], method='highs')
x_lp = [float(v) for v in res_lp.x]
Z_lp = -res_lp.fun
print(f"Z = {c1}x1 + {c2}x2 -> max; {a1[0]}x1+{a1[1]}x2<={b1}; {a2[0]}x1+{a2[1]}x2<={b2}")
print(f"Ослаблення: x = ({x_lp[0]:.4f}; {x_lp[1]:.4f}), Z_LP = {Z_lp:.4f} (верхня межа)")

## 2. Чому округлення не підходить

In [ ]:
def feasible(x1, x2):
    return a1[0]*x1 + a1[1]*x2 <= b1 + 1e-9 and a2[0]*x1 + a2[1]*x2 <= b2 + 1e-9

round_down = (math.floor(x_lp[0]), math.floor(x_lp[1]))
round_up = (math.ceil(x_lp[0]), math.ceil(x_lp[1]))
print('округлення вниз', round_down, '->', 'допустиме' if feasible(*round_down) else 'НЕдопустиме')
print('округлення вгору', round_up, '->', 'допустиме' if feasible(*round_up) else 'НЕдопустиме')

## 3. Метод гілок і меж (власна реалізація з протоколом вузлів)

In [ ]:
nodes_log = []
best = {"Z": -math.inf, "x": None}


def branch_and_bound(lo, hi, depth=0, parent=0):
    node_id = len(nodes_log) + 1
    r = linprog(c_min, A_ub=A, b_ub=b, bounds=list(zip(lo, hi)), method='highs')
    if not r.success:
        nodes_log.append(dict(node=node_id, parent=parent, depth=depth, Z=None, status='відсічено (недопустимий)'))
        return
    Z = -r.fun
    if Z <= best["Z"] + 1e-9 and best["x"] is not None:
        nodes_log.append(dict(node=node_id, parent=parent, depth=depth, Z=round(Z, 3), status='відсічено за межею'))
        return
    xr = [float(v) for v in r.x]
    frac = [i for i, v in enumerate(xr) if abs(v - round(v)) > 1e-6]
    if not frac:
        if Z > best["Z"]:
            best.update(Z=Z, x=xr)
        nodes_log.append(dict(node=node_id, parent=parent, depth=depth, Z=round(Z, 3), status=f'ЦІЛИЙ -> рекорд {xr}'))
        return
    i = frac[0]; v = xr[i]
    nodes_log.append(dict(node=node_id, parent=parent, depth=depth, Z=round(Z, 3), status=f'гілкування x{i+1}={v:.3f}'))
    hi1 = list(hi); hi1[i] = math.floor(v)
    branch_and_bound(lo, hi1, depth + 1, node_id)
    lo2 = list(lo); lo2[i] = math.ceil(v)
    branch_and_bound(lo2, hi, depth + 1, node_id)


branch_and_bound([0, 0], [None, None])
print(pd.DataFrame(nodes_log).to_string(index=False))
print(f"\nоптимум: x* = {best['x']}, Z* = {best['Z']:.0f}, вузлів: {len(nodes_log)}")

## 4. Перевірка через scipy.optimize.milp та розрив цілочисловості

In [ ]:
res_milp = milp(c_min, constraints=LinearConstraint(np.array(A), -np.inf, np.array(b)),
                integrality=np.array([1, 1]), bounds=Bounds(0, np.inf))
x_milp = [round(v) for v in res_milp.x]
Z_milp = -res_milp.fun
gap_pct = 100 * (Z_lp - best['Z']) / Z_lp
print(f"milp: x* = {x_milp}, Z* = {Z_milp:.0f}")
print(f"розрив цілочисловості: {gap_pct:.1f}%")

## 5. Задача про рюкзак: динамічне програмування vs жадібний алгоритм

In [ ]:
rng = np.random.default_rng(VARIANT)
w_items = rng.integers(3, 16, size=n_items).tolist()
p_items = rng.integers(4, 31, size=n_items).tolist()


def knapsack_dp(w, p, W):
    n = len(w)
    D = [[0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for cap in range(W + 1):
            D[i][cap] = D[i - 1][cap]
            if w[i - 1] <= cap:
                D[i][cap] = max(D[i][cap], D[i - 1][cap - w[i - 1]] + p[i - 1])
    items, cap = [], W
    for i in range(n, 0, -1):
        if D[i][cap] != D[i - 1][cap]:
            items.append(i - 1); cap -= w[i - 1]
    return D[n][W], sorted(items)


def knapsack_greedy(w, p, W):
    order = sorted(range(len(w)), key=lambda i: -p[i] / w[i])
    cap, val, items = W, 0, []
    for i in order:
        if w[i] <= cap:
            cap -= w[i]; val += p[i]; items.append(i)
    return val, sorted(items)


v_dp, items_dp = knapsack_dp(w_items, p_items, W)
v_gr, items_gr = knapsack_greedy(w_items, p_items, W)
print(f"ДП: цінність {v_dp}, предмети {[i+1 for i in items_dp]}")
print(f"жадібний: цінність {v_gr}, предмети {[i+1 for i in items_gr]}")
print(f"втрата жадібного: {100*(v_dp-v_gr)/v_dp:.1f}%")

## 6. Трудомісткість: ДП проти повного перебору

In [ ]:
def brute_force_knapsack(w, p, W):
    n = len(w); best_v = 0
    for mask in range(1 << n):
        tw = tp = 0
        for i in range(n):
            if mask & (1 << i):
                tw += w[i]; tp += p[i]
        if tw <= W and tp > best_v:
            best_v = tp
    return best_v


rows = []
for nn in (10, 15, 20, 22):
    rng2 = np.random.default_rng(VARIANT * 100 + nn)
    w2 = rng2.integers(3, 16, size=nn).tolist()
    p2 = rng2.integers(4, 31, size=nn).tolist()
    Wn = int(sum(w2) * 0.5)
    t0 = time.perf_counter(); knapsack_dp(w2, p2, Wn); t_dp = time.perf_counter() - t0
    t0 = time.perf_counter(); brute_force_knapsack(w2, p2, Wn); t_bf = time.perf_counter() - t0
    rows.append((nn, t_dp, t_bf))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy([r[0] for r in rows], [r[1] for r in rows], 'o-', label='ДП O(nW)')
ax.semilogy([r[0] for r in rows], [r[2] for r in rows], 's-', label='Повний перебір O(2^n)')
ax.set_xlabel('n'); ax.set_ylabel('час, с (лог.)'); ax.legend(); ax.grid(alpha=.3, which='both')
ax.set_title(f'Варіант {VARIANT}: трудомісткість ДП проти перебору')
fig.tight_layout(); fig